# Module 4 — Forward Kinematics using Denavit–Hartenberg Parameters
## Unit 8 — Mini Project: From Joints to the Fruit
### Lesson 8.3 — Verifying the Forward Kinematics

*Physical AI Curriculum · capstone notebook. Run **Kernel → Restart & Run All**.*

In [ ]:
import numpy as np
from functools import reduce
def dh(theta,d,a,al):
    ct,st,ca,sa=np.cos(theta),np.sin(theta),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1.0]])
def dh_fk(table,cfg):
    f=[dh(q,d,a,al) for (d,a,al,_),q in zip(table,cfg)]
    return reduce(lambda A,B:A@B,f,np.eye(4))
arm3=[(0.1,0,np.radians(90),'r'),(0,0.4,0,'r'),(0,0.3,0,'r')]
T=dh_fk(arm3,[0,np.radians(30),np.radians(60)])
print('T0^3 position',np.round(T[:3,3],3))

## Coding Exercise (§8)
Five independent verification checks → PASS/FAIL each.

In [ ]:
# YOUR CODE HERE
import numpy as np
def planar_reach(t1,t2,L1=0.4,L2=0.3):
    return np.hypot(L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2))
def verify(table):
    res={}
    # 1) zero config: in-plane reach = L1+L2 straightened, height includes riser
    T0=dh_fk(table,[0,0,0]); res['zero_config']=np.isclose(np.hypot(T0[0,3],T0[1,3]) if False else np.hypot(np.hypot(T0[0,3],T0[1,3]),T0[2,3]-0.1),0.7,atol=1e-6)
    # 2) planar reduction vs fk_planar
    Tp=dh_fk(table,[0,np.radians(30),np.radians(60)])
    reach=np.hypot(np.hypot(Tp[0,3],Tp[1,3]),Tp[2,3]-0.1)
    res['planar_reduction']=np.isclose(reach,planar_reach(np.radians(30),np.radians(60)),atol=1e-6)
    # 3) theta1 sweep = revolution (radius and height constant)
    rs=[];hs=[]
    for d1 in [0,45,90,180]:
        Ts=dh_fk(table,[np.radians(d1),np.radians(30),np.radians(60)])
        rs.append(np.hypot(Ts[0,3],Ts[1,3])); hs.append(Ts[2,3])
    res['theta1_revolution']=np.allclose(rs,rs[0],atol=1e-6) and np.allclose(hs,hs[0],atol=1e-6)
    # 4) SymPy = numeric
    import sympy as sp
    th=sp.symbols('t1 t2 t3',real=True)
    def sdh(t,d,a,al):
        return sp.Matrix([[sp.cos(t),-sp.sin(t)*sp.cos(al),sp.sin(t)*sp.sin(al),a*sp.cos(t)],[sp.sin(t),sp.cos(t)*sp.cos(al),-sp.cos(t)*sp.sin(al),a*sp.sin(t)],[0,sp.sin(al),sp.cos(al),d],[0,0,0,1]])
    Ts=sdh(th[0],sp.Rational(1,10),0,sp.pi/2)*sdh(th[1],0,sp.Rational(2,5),0)*sdh(th[2],0,sp.Rational(3,10),0)
    Tn=np.array(Ts.subs({th[0]:0,th[1]:sp.rad(30),th[2]:sp.rad(60)}).evalf(),float)
    res['sympy_match']=np.allclose(Tn,Tp,atol=1e-6)
    # 5) rotation validity
    R=Tp[:3,:3]; res['rotation_valid']=np.allclose(R.T@R,np.eye(3),atol=1e-9) and np.isclose(np.linalg.det(R),1,atol=1e-9)
    return res
r=verify(arm3)
for k,v in r.items(): print(('PASS' if v else 'FAIL'),k)
assert all(r.values())
print('all five checks passed → capstone FK verified.')

## Check your work

In [ ]:
import numpy as np
# corrupting alpha1 should break at least one check
bad=[(0.1,0,0,'r'),(0,0.4,0,'r'),(0,0.3,0,'r')]
assert not all(verify(bad).values())
print('All checks passed.')